# 台股學習資料稽核重現
只讀同目錄的已審閱結構快照；不重新連VM、不讀行情、不訓練、不計算績效。
原始查詢程式：repo 的 research/tw_a_learning_v1/audit_inventory.py；SQLite只使用sqlite_master與PRAGMA table_info。

In [1]:
import json
from pathlib import Path
e = json.loads(Path("audit_evidence.json").read_text(encoding="utf-8-sig"))
labels = e["labels"]
days = [d for d in labels["files"] if d["content_read"]]
assert all(d["date"] <= e["cutoff"] for d in days)
assert all(not d["content_read"] for d in labels["files"] if d["date"] > e["cutoff"])
assert sum(d["rows"] for d in days) == sum(p["rows"] for p in labels["profiles"]) == 6260
print("Historical label rows:", labels["rows"])

Historical label rows: 6260


In [2]:
cohorts = [(p["profile"], p["start_date"], p["end_date"], p["days"], p["rows"], p["common_stock_code_rows"]) for p in labels["profiles"]]
assert sum(p[-1] for p in cohorts) == 6224
for cohort in cohorts:
    print(cohort)

('208e5245843edcc4', '2026-09-11', '2026-09-18', 6, 2607, 2590)
('162cc665885f69b7', '2026-09-18', '2026-09-18', 1, 53, 53)
('18d13718d66141f1', '2026-09-22', '2026-09-22', 1, 424, 418)
('cc6815f1a2c2de2f', '2026-09-22', '2026-09-22', 1, 59, 59)
('06470428fd6d355f', '2026-09-24', '2026-09-24', 1, 470, 468)
('72a58274c3f1ff77', '2026-09-29', '2026-10-02', 4, 2647, 2636)


In [3]:
matched = sum(d["journal_join"]["matched_keys"] for d in days)
fresh = sum(d["journal_join"]["quote_time_causal_and_fresh"] for d in days)
assert matched == fresh == labels["valid_five_feature_rows"] == 6224
assert labels["duplicate_keys"] == 0
assert labels["explicit_label_available_at_rows"] == 0
print({"journal_matched": matched, "fresh_quote_timestamps": fresh, "raw_file_exists": labels["expanded_raw_file_exists_rows"]})

{'journal_matched': 6224, 'fresh_quote_timestamps': 6224, 'raw_file_exists': 3393}


In [4]:
snapshots = e["saved_cost_assumptions"]
assert len(snapshots) == len(days) == 12
assert all(s["costs"]["fee_rate"] == 0.0015 and s["costs"]["sell_tax_rate"] == 0.003 for s in snapshots)
assert not e["performance_calculated"] and not e["post_cutoff_content_opened"]
print("Cost snapshots checked:", len(snapshots))

Cost snapshots checked: 12
